# Chapter 14 Computational Lab
## Limit Theorems: The Law of Large Numbers and the Central Limit Theorem

This notebook accompanies Chapter 14 of *Probability Theory with Python and AI*.

Probability becomes especially powerful when we study long sequences of repeated observations. Two questions must be kept separate:

$$
\boxed{
\overline X_n\to\mu
}
$$

asks whether averages stabilize, while

$$
\boxed{
\frac{S_n-n\mu}{\sigma\sqrt n}
\Rightarrow N(0,1)
}
$$

asks about the shape of the remaining fluctuations after the correct centering and scaling.

### Learning goals

By the end of the lab you should be able to:

1. distinguish convergence almost surely, in $L^p$, in probability and in distribution;
2. use the correct one-way implication diagram among these modes;
3. construct counterexamples showing that the converse implications fail;
4. apply Chebyshev's inequality as a quantitative concentration bound;
5. prove and simulate the finite-variance weak law of large numbers;
6. understand Bernoulli's theorem for relative frequencies;
7. compute a rigorous Chebyshev sample-size guarantee for Bernoulli frequencies;
8. use the first and second Borel--Cantelli lemmas correctly;
9. distinguish convergence in probability from almost-sure convergence;
10. understand what Kolmogorov's maximal inequality controls;
11. understand the role of the Kolmogorov convergence criterion, Cesàro averaging and Kronecker's lemma in the strong law;
12. state the strong law under the optimal first-moment assumption $\mathbb E|X_1|<\infty$;
13. explain why infinite variance does not automatically destroy the law of large numbers;
14. understand strong consistency of the sample variance;
15. use characteristic functions to establish distributional convergence;
16. understand the known-limit form of Lévy's continuity theorem;
17. apply Slutsky's theorem;
18. derive the local second-order expansion of a centered unit-variance characteristic function;
19. follow the characteristic-function proof of the classical CLT;
20. use the studentized CLT;
21. understand what Berry--Esseen adds beyond the qualitative CLT;
22. derive the binomial normal limit and use continuity correction;
23. distinguish the LLN, CLT and exact normality in the Gaussian parent case;
24. explain why standard Cauchy averages do not stabilize;
25. audit AI-generated statements about limit theorems.

> **Central distinction.** The law of large numbers describes stabilization of averages. The central limit theorem describes the scaled fluctuations around that stabilized value.


## 0. Setup

The notebook uses direct formulas and simulations. Computation illustrates the theorems but does not replace their proofs.


In [ ]:
from math import comb
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def normal_cdf(x):
    return 0.5*(1 + math.erf(x/math.sqrt(2)))


def normal_pdf(x):
    x = np.asarray(x, dtype=float)
    return np.exp(-x*x/2)/math.sqrt(2*math.pi)


def binomial_pmf(k, n, p):
    if not isinstance(k, int) or k < 0 or k > n:
        return 0.0
    return comb(n, k)*(p**k)*((1-p)**(n-k))


def binomial_interval_probability(a, b, n, p):
    a = max(0, int(a))
    b = min(n, int(b))
    if a > b:
        return 0.0
    return sum(binomial_pmf(k, n, p) for k in range(a, b+1))


def chebyshev_mean_bound(variance, n, epsilon):
    return min(1.0, variance/(n*epsilon*epsilon))


def bernoulli_universal_sample_size(epsilon, eta):
    return math.ceil(1/(4*epsilon*epsilon*eta))


def standardized_sum(samples, mu, sigma):
    samples = np.asarray(samples, dtype=float)
    n = samples.shape[1]
    return (samples.sum(axis=1)-n*mu)/(sigma*math.sqrt(n))


def empirical_cdf_at(sample, x):
    sample = np.asarray(sample, dtype=float)
    return np.mean(sample <= x)


def sample_variance_unbiased(x):
    return np.var(np.asarray(x, dtype=float), ddof=1)


def standard_cauchy_cf(t):
    return np.exp(-np.abs(np.asarray(t, dtype=float)))


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Limit-theorem tools are ready."
    "</div>"
))


## 1. Why limit theorems are needed

Let

$$
S_n=X_1+\cdots+X_n,
\qquad
\overline X_n=\frac{S_n}{n}.
$$

For i.i.d. variables with

$$
\mathbb E[X_1]=\mu,
\qquad
\operatorname{Var}(X_1)=\sigma^2<\infty,
$$

we have

$$
\boxed{
\mathbb E[\overline X_n]=\mu,
\qquad
\operatorname{Var}(\overline X_n)=\frac{\sigma^2}{n}.
}
$$

The sample mean becomes increasingly concentrated around $\mu$.


At the same time,

$$
\operatorname{Var}(S_n-n\mu)=n\sigma^2.
$$

Therefore the typical fluctuation of $S_n-n\mu$ has order $\sqrt n$, suggesting the standardized variable

$$
\boxed{
Z_n=
\frac{S_n-n\mu}{\sigma\sqrt n}.
}
$$

The LLN studies $\overline X_n$.

The CLT studies $Z_n$.


In [ ]:
scale_n = widgets.IntSlider(value=100, min=1, max=1000, description="n")
scale_sigma = widgets.FloatSlider(value=2, min=0.1, max=5, step=0.1, description="sigma")
scale_output = widgets.Output()


def update_scaling(*_):
    with scale_output:
        clear_output(wait=True)

        n = scale_n.value
        sigma = scale_sigma.value

        display(Math(
            r"\operatorname{SD}(\overline X_n)="
            + f"{sigma/math.sqrt(n):.6f}"
        ))
        display(Math(
            r"\operatorname{SD}(S_n-n\mu)="
            + f"{sigma*math.sqrt(n):.6f}"
        ))


for control in (scale_n, scale_sigma):
    control.observe(update_scaling, names="value")

display(widgets.VBox([
    widgets.HBox([scale_n, scale_sigma]),
    scale_output,
]))
update_scaling()


## 2. Four modes of convergence

### Convergence in probability

$$
\boxed{
X_n\xrightarrow{P}X
}
$$

means that for every $\varepsilon>0$,

$$
P(|X_n-X|>\varepsilon)\to0.
$$

Fixed-size errors become unlikely.


### Convergence in distribution

Let $F_n$ and $F$ be the cdfs of $X_n$ and $X$.

$$
\boxed{
X_n\xrightarrow{d}X
}
$$

means

$$
F_n(x)\to F(x)
$$

at every continuity point $x$ of $F$.

This is convergence of laws, not pathwise convergence.


### Almost-sure convergence

$$
\boxed{
X_n\xrightarrow{\mathrm{a.s.}}X
}
$$

means

$$
P\left(
\omega:
X_n(\omega)\to X(\omega)
\right)=1.
$$

Outside one fixed null set, the sample path converges pointwise.


### $L^p$ convergence

For $p>0$,

$$
\boxed{
X_n\to X\text{ in }L^p
}
$$

means

$$
\mathbb E[|X_n-X|^p]\to0.
$$

For $p=2$ this is mean-square convergence.


### Basic implication diagram

The chapter proves

$$
\boxed{
X_n\xrightarrow{\mathrm{a.s.}}X
\Longrightarrow
X_n\xrightarrow{P}X
\Longrightarrow
X_n\xrightarrow{d}X,
}
$$

and

$$
\boxed{
X_n\to X\text{ in }L^p
\Longrightarrow
X_n\xrightarrow{P}X.
}
$$

The arrows are generally one-way.


In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
ax.axis("off")

ax.text(0.08,0.70,"Almost sure",ha="center",va="center",bbox=dict(boxstyle="round",fc="none"))
ax.text(0.08,0.25,"L^p",ha="center",va="center",bbox=dict(boxstyle="round",fc="none"))
ax.text(0.50,0.48,"In probability",ha="center",va="center",bbox=dict(boxstyle="round",fc="none"))
ax.text(0.88,0.48,"In distribution",ha="center",va="center",bbox=dict(boxstyle="round",fc="none"))

ax.annotate("",xy=(0.43,0.52),xytext=(0.16,0.68),arrowprops=dict(arrowstyle="->"))
ax.annotate("",xy=(0.43,0.44),xytext=(0.16,0.28),arrowprops=dict(arrowstyle="->"))
ax.annotate("",xy=(0.80,0.48),xytext=(0.60,0.48),arrowprops=dict(arrowstyle="->"))

ax.set_title("One-way implications among modes of convergence")
plt.show()


### Deterministic approximation

Let

$$
X_n=X+\frac1n.
$$

For every fixed $\varepsilon>0$,

$$
P(|X_n-X|>\varepsilon)=0
$$

for all sufficiently large $n$.

Hence

$$
X_n\xrightarrow{P}X.
$$


### Almost sure does not imply $L^1$

On $(0,1)$ with uniform probability, define

$$
X_n(\omega)
=
n\mathbf 1_{(0,1/n)}(\omega).
$$

For every fixed $\omega>0$, eventually $X_n(\omega)=0$, so

$$
X_n\xrightarrow{\mathrm{a.s.}}0.
$$

But

$$
\boxed{
\mathbb E|X_n|=1
}
$$

for every $n$.

Therefore almost-sure convergence alone does not justify convergence of expectations.


In [ ]:
as_n = widgets.IntSlider(value=20, min=1, max=100, description="n")
as_output = widgets.Output()


def update_as_not_l1(*_):
    with as_output:
        clear_output(wait=True)

        n = as_n.value
        omega = np.linspace(0.0001,1,1200)
        Xn = np.where(omega < 1/n, n, 0)

        fig, ax = plt.subplots(figsize=(8,3.2))
        ax.plot(omega,Xn)
        ax.set_xlabel("omega")
        ax.set_ylabel("X_n(omega)")
        ax.set_title("Almost sure convergence to 0 but constant L1 norm")
        plt.show()

        display(Math(r"\mathbb E|X_n|=1"))


as_n.observe(update_as_not_l1, names="value")
display(widgets.VBox([as_n,as_output]))
update_as_not_l1()


### Distribution convergence does not imply probability convergence

Let $X$ take the values $-1$ and $1$ with probability $1/2$ each, and define

$$
X_n=(-1)^nX.
$$

Every $X_n$ has the same law as $X$, so

$$
X_n\xrightarrow{d}X.
$$

But for every odd $n$,

$$
|X_n-X|=2,
$$

and therefore

$$
P(|X_n-X|>1)=1.
$$


### Constant distributional limit

If

$$
X_n\xrightarrow{d}c
$$

for a constant $c$, then in fact

$$
\boxed{
X_n\xrightarrow{P}c.
}
$$

This is an important special converse.


## 3. Chebyshev's inequality

If $X$ has finite mean $\mu$ and variance $\sigma^2$, then for every $\varepsilon>0$,

$$
\boxed{
P(|X-\mu|\ge\varepsilon)
\le
\frac{\sigma^2}{\varepsilon^2}.
}
$$

For $\sigma>0$,

$$
P(|X-\mu|\ge k\sigma)\le\frac1{k^2}.
$$

The inequality is universal but may be conservative.


### How conservative can it be?

If

$$
X\sim N(100,15^2),
$$

Chebyshev gives

$$
P(|X-100|\ge30)\le\frac14.
$$

The exact normal probability is

$$
2(1-\Phi(2))
\approx0.0455.
$$


In [ ]:
cheb_k = widgets.FloatSlider(value=2, min=0.5, max=5, step=0.1, description="k")
cheb_output = widgets.Output()


def update_chebyshev_normal(*_):
    with cheb_output:
        clear_output(wait=True)

        k = cheb_k.value
        bound = min(1.0,1/(k*k))
        exact = 2*(1-normal_cdf(k))

        display(Math(r"\text{Chebyshev bound}=" + f"{bound:.6f}"))
        display(Math(r"\text{exact normal tail}=" + f"{exact:.6f}"))


cheb_k.observe(update_chebyshev_normal, names="value")
display(widgets.VBox([cheb_k,cheb_output]))
update_chebyshev_normal()


## 4. Weak Law of Large Numbers: finite-variance form

Let $X_1,X_2,\ldots$ be i.i.d. with

$$
\mathbb E[X_1]=\mu,
\qquad
\operatorname{Var}(X_1)=\sigma^2<\infty.
$$

Then

$$
\boxed{
\overline X_n\xrightarrow{P}\mu.
}
$$

Indeed,

$$
\operatorname{Var}(\overline X_n)
=
\frac{\sigma^2}{n},
$$

and Chebyshev gives

$$
\boxed{
P(|\overline X_n-\mu|>\varepsilon)
\le
\frac{\sigma^2}{n\varepsilon^2}.
}
$$


### Poisson sample averages

If

$$
X_i\stackrel{\mathrm{i.i.d.}}{\sim}\operatorname{Poisson}(3),
$$

then

$$
\mu=3,
\qquad
\sigma^2=3.
$$

Thus

$$
P(|\overline X_n-3|>0.5)
\le
\frac{12}{n}.
$$


In [ ]:
wlln_n = widgets.IntSlider(value=100, min=10, max=5000, step=10, description="n")
wlln_eps = widgets.FloatSlider(value=0.5, min=0.1, max=2, step=0.1, description="epsilon")
wlln_output = widgets.Output()


def update_wlln_bound(*_):
    with wlln_output:
        clear_output(wait=True)

        n = wlln_n.value
        eps = wlln_eps.value
        bound = chebyshev_mean_bound(3,n,eps)

        display(Math(
            r"P(|\overline X_n-3|>\varepsilon)\le"
            + f"{bound:.6f}"
        ))


for control in (wlln_n,wlln_eps):
    control.observe(update_wlln_bound, names="value")

display(widgets.VBox([
    widgets.HBox([wlln_n,wlln_eps]),
    wlln_output,
]))
update_wlln_bound()


In [ ]:
wlln_sim_n = widgets.IntSlider(value=100, min=10, max=2000, step=10, description="n")
wlln_sim_R = widgets.IntSlider(value=20000, min=1000, max=100000, step=1000, description="reps")
wlln_sim_output = widgets.Output()


def update_wlln_sim(*_):
    with wlln_sim_output:
        clear_output(wait=True)

        n = wlln_sim_n.value
        R = wlln_sim_R.value
        eps = 0.5

        rng = np.random.default_rng(2026)
        samples = rng.poisson(3,size=(R,n))
        means = samples.mean(axis=1)

        empirical = np.mean(np.abs(means-3)>eps)
        bound = chebyshev_mean_bound(3,n,eps)

        display(Math(r"\widehat P(|\overline X_n-3|>0.5)=" + f"{empirical:.6f}"))
        display(Math(r"\text{Chebyshev bound}=" + f"{bound:.6f}"))


for control in (wlln_sim_n,wlln_sim_R):
    control.observe(update_wlln_sim, names="value")

display(widgets.VBox([
    widgets.HBox([wlln_sim_n,wlln_sim_R]),
    wlln_sim_output,
]))
update_wlln_sim()


### Variance criterion for non-identically distributed averages

If independent $X_j$ have finite variances and

$$
\frac1{n^2}
\sum_{j=1}^n
\operatorname{Var}(X_j)
\to0,
$$

then

$$
\boxed{
\frac1n
\sum_{j=1}^n
\left(
X_j-\mathbb E[X_j]
\right)
\xrightarrow{P}0.
}
$$

A uniform variance bound is sufficient because

$$
\frac1{n^2}
\sum_{j=1}^n
\operatorname{Var}(X_j)
\le
\frac Cn.
$$


## 5. Bernoulli's theorem and relative frequencies

Let $I_1,I_2,\ldots$ be independent Bernoulli$(p)$ variables and

$$
N_n=I_1+\cdots+I_n.
$$

Then

$$
\boxed{
\frac{N_n}{n}\xrightarrow{P}p.
}
$$

More precisely,

$$
P\left(
\left|
\frac{N_n}{n}-p
\right|
\ge\varepsilon
\right)
\le
\frac{p(1-p)}{n\varepsilon^2}
\le
\frac1{4n\varepsilon^2}.
$$


In [ ]:
bern_p = widgets.FloatSlider(value=0.3, min=0.01, max=0.99, step=0.01, description="p")
bern_n = widgets.IntSlider(value=200, min=10, max=5000, step=10, description="n")
bern_eps = widgets.FloatSlider(value=0.05, min=0.01, max=0.2, step=0.01, description="epsilon")
bern_output = widgets.Output()


def update_bernoulli_bound(*_):
    with bern_output:
        clear_output(wait=True)

        p = bern_p.value
        n = bern_n.value
        eps = bern_eps.value

        specific = min(1.0,p*(1-p)/(n*eps*eps))
        universal = min(1.0,1/(4*n*eps*eps))

        display(Math(r"\text{specific bound}=" + f"{specific:.6f}"))
        display(Math(r"\text{universal bound}=" + f"{universal:.6f}"))


for control in (bern_p,bern_n,bern_eps):
    control.observe(update_bernoulli_bound, names="value")

display(widgets.VBox([
    widgets.HBox([bern_p,bern_n,bern_eps]),
    bern_output,
]))
update_bernoulli_bound()


## 6. Historical problem: how many observations make a frequency trustworthy?

Suppose we want

$$
P\left(
\left|
\frac{N_n}{n}-p
\right|
<\varepsilon
\right)
\ge
1-\eta
$$

for every $p\in[0,1]$.

The universal Chebyshev--Bernoulli bound gives the sufficient condition

$$
\boxed{
n\ge
\frac1{4\varepsilon^2\eta}.
}
$$

This is a rigorous finite-sample guarantee, not necessarily a minimal sample size.


In [ ]:
hist_eps = widgets.FloatSlider(value=0.05, min=0.01, max=0.20, step=0.01, description="epsilon")
hist_eta = widgets.FloatSlider(value=0.05, min=0.01, max=0.20, step=0.01, description="eta")
hist_output = widgets.Output()


def update_historical_sample_size(*_):
    with hist_output:
        clear_output(wait=True)

        eps = hist_eps.value
        eta = hist_eta.value
        n = bernoulli_universal_sample_size(eps,eta)

        display(Math(
            r"n\ge\left\lceil\frac1{4\varepsilon^2\eta}\right\rceil="
            + f"{n}"
        ))


for control in (hist_eps,hist_eta):
    control.observe(update_historical_sample_size, names="value")

display(widgets.VBox([
    widgets.HBox([hist_eps,hist_eta]),
    hist_output,
]))
update_historical_sample_size()


For

$$
\varepsilon=0.05,
\qquad
\eta=0.05,
$$

the sufficient universal sample size is

$$
\boxed{
n=2000.
}
$$

The historical point is Bernoulli's frequency question. The displayed bound is a modern Chebyshev-based solution.


## 7. Borel--Cantelli and pathwise statements

### First Borel--Cantelli lemma

If

$$
\sum_{n=1}^\infty P(A_n)<\infty,
$$

then

$$
\boxed{
P(A_n\text{ infinitely often})=0.
}
$$

No independence assumption is required.


### Second Borel--Cantelli lemma

If the events are mutually independent and

$$
\sum_{n=1}^{\infty}P(A_n)=\infty,
$$

then

$$
\boxed{
P(A_n\text{ infinitely often})=1.
}
$$

Here independence is essential.


### Convergence in probability need not be almost sure

Let $A_n$ be independent with

$$
P(A_n)=\frac1n,
$$

and let

$$
X_n=\mathbf 1_{A_n}.
$$

Then

$$
P(|X_n|>1/2)=\frac1n\to0,
$$

so

$$
X_n\xrightarrow{P}0.
$$

But

$$
\sum_nP(A_n)=\infty.
$$

The second Borel--Cantelli lemma gives infinitely many $X_n=1$ almost surely, so

$$
X_n\not\to0
\quad\text{a.s.}
$$


## 8. Kolmogorov's maximal inequality

Let $X_1,\ldots,X_n$ be independent, mean-zero variables with finite variances, and let

$$
S_k=X_1+\cdots+X_k.
$$

Then for every $\lambda>0$,

$$
\boxed{
P\left(
\max_{1\le k\le n}|S_k|
\ge\lambda
\right)
\le
\frac{\operatorname{Var}(S_n)}{\lambda^2}.
}
$$

Unlike ordinary Chebyshev, this controls **all partial sums simultaneously**.


### Simple random walk

For independent Rademacher increments,

$$
P(X_k=\pm1)=\frac12,
$$

we have

$$
\operatorname{Var}(S_n)=n.
$$

Therefore

$$
\boxed{
P\left(
\max_{k\le n}|S_k|
\ge3\sqrt n
\right)
\le
\frac19.
}
$$


In [ ]:
kol_n = widgets.IntSlider(value=100, min=10, max=2000, step=10, description="n")
kol_R = widgets.IntSlider(value=10000, min=1000, max=50000, step=1000, description="reps")
kol_a = widgets.FloatSlider(value=3, min=1, max=5, step=0.25, description="a")
kol_output = widgets.Output()


def update_kolmogorov_sim(*_):
    with kol_output:
        clear_output(wait=True)

        n = kol_n.value
        R = kol_R.value
        a = kol_a.value

        rng = np.random.default_rng(2026)
        steps = rng.choice([-1,1],size=(R,n))
        paths = np.cumsum(steps,axis=1)
        event = np.max(np.abs(paths),axis=1) >= a*math.sqrt(n)

        empirical = np.mean(event)
        bound = min(1.0,1/(a*a))

        display(Math(r"\widehat P(\max_{k\le n}|S_k|\ge a\sqrt n)=" + f"{empirical:.6f}"))
        display(Math(r"\text{Kolmogorov bound}=" + f"{bound:.6f}"))


for control in (kol_n,kol_R,kol_a):
    control.observe(update_kolmogorov_sim, names="value")

display(widgets.VBox([
    widgets.HBox([kol_n,kol_R,kol_a]),
    kol_output,
]))
update_kolmogorov_sim()


## 9. Kolmogorov convergence criterion

If $Z_1,Z_2,\ldots$ are independent, mean zero, have finite variances, and

$$
\sum_{n=1}^{\infty}\operatorname{Var}(Z_n)<\infty,
$$

then

$$
\boxed{
\sum_{n=1}^{\infty}Z_n
\text{ converges almost surely}.
}
$$


### Random harmonic series

Let $\varepsilon_n$ be independent Rademacher variables and

$$
Z_n=\frac{\varepsilon_n}{n}.
$$

Then

$$
\sum_n\operatorname{Var}(Z_n)
=
\sum_n\frac1{n^2}
<
\infty.
$$

Hence

$$
\boxed{
\sum_{n=1}^{\infty}
\frac{\varepsilon_n}{n}
}
$$

converges almost surely.

Random signs can create convergence even though the deterministic harmonic series diverges.


In [ ]:
rh_N = widgets.IntSlider(value=5000, min=100, max=30000, step=100, description="N")
rh_seed = widgets.IntSlider(value=2026, min=0, max=5000, description="seed")
rh_output = widgets.Output()


def update_random_harmonic(*_):
    with rh_output:
        clear_output(wait=True)

        N = rh_N.value
        seed = rh_seed.value

        rng = np.random.default_rng(seed)
        eps = rng.choice([-1,1],size=N)
        partial = np.cumsum(eps/np.arange(1,N+1))

        fig, ax = plt.subplots(figsize=(8,3.5))
        ax.plot(np.arange(1,N+1),partial)
        ax.set_xlabel("n")
        ax.set_ylabel("partial sum")
        ax.set_title("Random harmonic series partial sums")
        plt.show()

        display(Markdown(f"Final partial sum: **{partial[-1]:.6f}**"))


for control in (rh_N,rh_seed):
    control.observe(update_random_harmonic, names="value")

display(widgets.VBox([
    widgets.HBox([rh_N,rh_seed]),
    rh_output,
]))
update_random_harmonic()


### Cesàro averaging and Kronecker's lemma

The proof of the strong law uses two deterministic tools.

If $a_n\to a$, then

$$
\boxed{
\frac1n\sum_{k=1}^na_k\to a.
}
$$

This is the Cesàro averaging lemma.

If

$$
\sum_{n=1}^\infty\frac{a_n}{n}
$$

converges, then

$$
\boxed{
\frac1n\sum_{k=1}^na_k\to0.
}
$$

This is Kronecker's lemma.


## 10. Strong Law of Large Numbers

Let $X_1,X_2,\ldots$ be i.i.d. real random variables with

$$
\boxed{
\mathbb E|X_1|<\infty.
}
$$

If

$$
\mu=\mathbb E[X_1],
$$

then

$$
\boxed{
\overline X_n
\xrightarrow{\mathrm{a.s.}}
\mu.
}
$$

The strong law gives pathwise stabilization under only a finite first absolute moment.


### Proof architecture

The chapter's proof uses truncation:

$$
Y_n
=
X_n\mathbf 1_{\{|X_n|\le n\}}.
$$

The main steps are:

1. use the first Borel--Cantelli lemma to show $X_n=Y_n$ eventually almost surely;
2. prove summability of the variances of

$$
Z_n=
\frac{Y_n-\mathbb E[Y_n]}{n};
$$

3. apply the Kolmogorov convergence criterion;
4. apply Kronecker's lemma;
5. use dominated convergence and Cesàro averaging for the means.

This is stronger than the elementary Chebyshev proof of the weak law.


In [ ]:
slln_p = widgets.FloatSlider(value=0.37, min=0.01, max=0.99, step=0.01, description="p")
slln_N = widgets.IntSlider(value=20000, min=1000, max=100000, step=1000, description="N")
slln_output = widgets.Output()


def update_slln_path(*_):
    with slln_output:
        clear_output(wait=True)

        p = slln_p.value
        N = slln_N.value

        rng = np.random.default_rng(2026)
        x = rng.binomial(1,p,size=N)
        running = np.cumsum(x)/np.arange(1,N+1)

        fig, ax = plt.subplots(figsize=(8,3.6))
        ax.plot(np.arange(1,N+1),running,label="running frequency")
        ax.axhline(p,linestyle="--",label="p")
        ax.set_xlabel("n")
        ax.set_ylabel("sample mean")
        ax.set_title("Strong-law picture along one Bernoulli sample path")
        ax.legend()
        plt.show()

        display(Math(r"\overline X_N=" + f"{running[-1]:.6f}"))


for control in (slln_p,slln_N):
    control.observe(update_slln_path, names="value")

display(widgets.VBox([
    widgets.HBox([slln_p,slln_N]),
    slln_output,
]))
update_slln_path()


### Strong law for relative frequencies

For i.i.d. Bernoulli$(p)$ variables,

$$
\boxed{
\frac{N_n}{n}
\xrightarrow{\mathrm{a.s.}}
p.
}
$$

This is stronger than the earlier Bernoulli weak-law statement.


## 11. Finite mean but infinite variance

Consider the Pareto density

$$
f(x)
=
\frac32x^{-5/2}
\mathbf 1_{[1,\infty)}(x).
$$

Then

$$
\mathbb E[X]=3<\infty,
$$

but

$$
\mathbb E[X^2]=\infty.
$$

Therefore the strong law still gives

$$
\boxed{
\overline X_n\xrightarrow{\mathrm{a.s.}}3.
}
$$

However, the finite-variance Chebyshev proof of the WLLN and the classical finite-variance CLT from this chapter do not apply.


In [ ]:
pareto_N = widgets.IntSlider(value=20000, min=1000, max=100000, step=1000, description="N")
pareto_seed = widgets.IntSlider(value=2026, min=0, max=5000, description="seed")
pareto_output = widgets.Output()


def update_pareto_path(*_):
    with pareto_output:
        clear_output(wait=True)

        N = pareto_N.value
        seed = pareto_seed.value
        rng = np.random.default_rng(seed)

        # numpy.pareto(a) has support y>=0 with tail (1+y)^(-a).
        # X=1+Y gives Pareto scale 1, shape alpha=3/2.
        x = 1 + rng.pareto(1.5,size=N)
        running = np.cumsum(x)/np.arange(1,N+1)

        fig, ax = plt.subplots(figsize=(8,3.6))
        ax.plot(np.arange(1,N+1),running)
        ax.axhline(3,linestyle="--")
        ax.set_xlabel("n")
        ax.set_ylabel("running mean")
        ax.set_title("Finite-mean, infinite-variance Pareto sample mean")
        plt.show()

        display(Markdown(f"Final running mean: **{running[-1]:.6f}**"))
        display(Markdown(f"Largest observation: **{np.max(x):.6g}**"))


for control in (pareto_N,pareto_seed):
    control.observe(update_pareto_path, names="value")

display(widgets.VBox([
    widgets.HBox([pareto_N,pareto_seed]),
    pareto_output,
]))
update_pareto_path()


## 12. Strong consistency of the sample variance

Suppose the i.i.d. variables have finite variance $\sigma^2$.

Define

$$
\widehat\sigma_n^2
=
\frac1{n-1}
\sum_{k=1}^n
(X_k-\overline X_n)^2.
$$

Then

$$
\boxed{
\widehat\sigma_n^2
\xrightarrow{\mathrm{a.s.}}
\sigma^2,
}
$$

and

$$
\boxed{
\widehat\sigma_n
\xrightarrow{\mathrm{a.s.}}
\sigma.
}
$$


The proof uses

$$
\frac1n
\sum_{k=1}^n
(X_k-\overline X_n)^2
=
\frac1n
\sum_{k=1}^nX_k^2
-
\overline X_n^2,
$$

and applies the strong law both to $X_k$ and to $X_k^2$.


In [ ]:
sv_N = widgets.IntSlider(value=5000, min=100, max=50000, step=100, description="N")
sv_output = widgets.Output()


def update_sample_variance(*_):
    with sv_output:
        clear_output(wait=True)

        N = sv_N.value
        rng = np.random.default_rng(2026)
        x = rng.binomial(1,0.3,size=N)

        checkpoints = np.unique(np.linspace(10,N,180,dtype=int))
        estimates = np.array([
            np.var(x[:n],ddof=1)
            for n in checkpoints
        ])

        fig, ax = plt.subplots(figsize=(8,3.4))
        ax.plot(checkpoints,estimates)
        ax.axhline(0.3*0.7,linestyle="--")
        ax.set_xlabel("n")
        ax.set_ylabel("sample variance")
        ax.set_title("Strong consistency of Bernoulli sample variance")
        plt.show()

        display(Math(r"\widehat\sigma_N^2=" + f"{estimates[-1]:.6f}"))
        display(Math(r"p(1-p)=0.21"))


sv_N.observe(update_sample_variance, names="value")
display(widgets.VBox([sv_N,sv_output]))
update_sample_variance()


## 13. Characteristic functions and distributional limits

The central limit theorem is proved using characteristic functions.

The key bridge is not merely uniqueness, but continuity of the transform-to-law correspondence in the correct direction.


### Scheffé's lemma for probability densities

If $f_n$ and $f$ are probability densities and

$$
f_n(x)\to f(x)
$$

for almost every $x$, then

$$
\boxed{
\int_{\mathbb R}|f_n-f|\to0.
}
$$

Consequently, the corresponding cdfs converge uniformly.


In [ ]:
scheffe_n = widgets.IntSlider(value=5, min=1, max=100, description="n")
scheffe_output = widgets.Output()


def update_scheffe(*_):
    with scheffe_output:
        clear_output(wait=True)

        n = scheffe_n.value
        sigma_n = math.sqrt(1+1/n)

        x = np.linspace(-6,6,30000)
        fn = np.exp(-x*x/(2*sigma_n*sigma_n))/(sigma_n*math.sqrt(2*math.pi))
        f = normal_pdf(x)

        if hasattr(np,"trapezoid"):
            l1 = np.trapezoid(np.abs(fn-f),x)
        else:
            l1 = np.trapz(np.abs(fn-f),x)

        fig, ax = plt.subplots(figsize=(8,3.3))
        ax.plot(x,f,label="N(0,1)")
        ax.plot(x,fn,linestyle="--",label=f"N(0,{1+1/n:.3f})")
        ax.legend()
        ax.set_title("Nearby normal densities")
        plt.show()

        display(Math(r"\int|f_n-f|\approx" + f"{l1:.8f}"))


scheffe_n.observe(update_scheffe, names="value")
display(widgets.VBox([scheffe_n,scheffe_output]))
update_scheffe()


## 14. Lévy's continuity theorem: known-limit form

Suppose $X_n$ has characteristic function $\varphi_n$ and $X$ has characteristic function $\varphi$.

If

$$
\varphi_n(t)\to\varphi(t)
$$

for every real $t$, then

$$
\boxed{
X_n\xrightarrow{d}X.
}
$$

The proof in the chapter reuses the Gaussian-smoothing argument from Chapter 13.


### Poisson parameters converging

If

$$
X_n\sim\operatorname{Poisson}(\lambda_n)
$$

and

$$
\lambda_n\to\lambda,
$$

then

$$
\varphi_{X_n}(t)
=
\exp\left(
\lambda_n(e^{it}-1)
\right)
\to
\exp\left(
\lambda(e^{it}-1)
\right).
$$

Hence

$$
\boxed{
X_n\xrightarrow{d}\operatorname{Poisson}(\lambda).
}
$$


In [ ]:
levy_n = widgets.IntSlider(value=5, min=1, max=100, description="n")
levy_output = widgets.Output()


def update_levy_poisson(*_):
    with levy_output:
        clear_output(wait=True)

        n = levy_n.value
        lam_n = 2 + 1/n
        lam = 2

        t = np.linspace(-math.pi,math.pi,600)
        phi_n = np.exp(lam_n*(np.exp(1j*t)-1))
        phi = np.exp(lam*(np.exp(1j*t)-1))

        display(Markdown(
            f"Maximum CF difference on the grid: **{np.max(np.abs(phi_n-phi)):.6g}**"
        ))


levy_n.observe(update_levy_poisson, names="value")
display(widgets.VBox([levy_n,levy_output]))
update_levy_poisson()


## 15. Slutsky's theorem

If

$$
X_n\xrightarrow{d}X,
\qquad
Y_n\xrightarrow{P}c,
$$

where $c$ is constant, then

$$
\boxed{
X_n+Y_n\xrightarrow{d}X+c,
}
$$

$$
\boxed{
X_nY_n\xrightarrow{d}cX,
}
$$

and, if $c\ne0$,

$$
\boxed{
\frac{X_n}{Y_n}
\xrightarrow{d}
\frac Xc.
}
$$

This theorem justifies replacing unknown constants by consistent estimators in asymptotic normalizations.


### Simple example

If

$$
X_n\xrightarrow{d}N(0,1),
$$

and

$$
Y_n\xrightarrow{P}2,
$$

then

$$
X_n+Y_n\xrightarrow{d}N(2,1),
$$

and

$$
\frac{X_n}{Y_n}
\xrightarrow{d}
N\left(0,\frac14\right).
$$


## 16. Local expansion of a characteristic function

If

$$
\mathbb E[Y]=0,
\qquad
\mathbb E[Y^2]=1,
$$

then

$$
\boxed{
\varphi_Y(u)
=
1-\frac{u^2}{2}+o(u^2)
\qquad
(u\to0).
}
$$

Only a finite second moment is required.


The proof writes

$$
e^{iuY}
=
1+iuY-\frac{u^2Y^2}{2}
+
u^2Y^2h(uY),
$$

where $h(x)\to0$ and $h$ is bounded.

Then dominated convergence gives

$$
\mathbb E[Y^2h(uY)]\to0.
$$


### Rademacher example

If

$$
P(Y=1)=P(Y=-1)=\frac12,
$$

then

$$
\varphi_Y(u)=\cos u.
$$

The ordinary Taylor expansion gives

$$
\cos u
=
1-\frac{u^2}{2}+o(u^2),
$$

matching the abstract lemma.


In [ ]:
local_u = widgets.FloatSlider(value=0.5, min=0.02, max=2, step=0.02, description="u")
local_output = widgets.Output()


def update_local_expansion(*_):
    with local_output:
        clear_output(wait=True)

        u = local_u.value
        exact = math.cos(u)
        quadratic = 1-u*u/2
        remainder_ratio = abs(exact-quadratic)/(u*u)

        display(Math(r"\cos u=" + f"{exact:.10f}"))
        display(Math(r"1-u^2/2=" + f"{quadratic:.10f}"))
        display(Math(
            r"\frac{|\cos u-(1-u^2/2)|}{u^2}="
            + f"{remainder_ratio:.8f}"
        ))


local_u.observe(update_local_expansion, names="value")
display(widgets.VBox([local_u,local_output]))
update_local_expansion()


### Elementary complex exponentiation limit

If

$$
z_n\to0,
\qquad
nz_n\to z,
\qquad
n|z_n|^2\to0,
$$

then

$$
\boxed{
(1+z_n)^n\to e^z.
}
$$

This is the final analytic ingredient in the chapter's CLT proof.


## 17. Central Limit Theorem

Let $X_1,X_2,\ldots$ be i.i.d. with

$$
\mathbb E[X_1]=\mu,
$$

and

$$
0<\operatorname{Var}(X_1)=\sigma^2<\infty.
$$

Then

$$
\boxed{
\frac{S_n-n\mu}{\sigma\sqrt n}
\xrightarrow{d}
N(0,1).
}
$$

Equivalently,

$$
\boxed{
\frac{\sqrt n(\overline X_n-\mu)}{\sigma}
\xrightarrow{d}
N(0,1).
}
$$


### Characteristic-function proof architecture

Standardize one observation:

$$
Y_j=\frac{X_j-\mu}{\sigma}.
$$

Then

$$
\mathbb E[Y_j]=0,
\qquad
\mathbb E[Y_j^2]=1.
$$

The local expansion gives

$$
\varphi(u)
=
1-\frac{u^2}{2}+o(u^2).
$$

For

$$
Z_n=
\frac{Y_1+\cdots+Y_n}{\sqrt n},
$$

independence yields

$$
\varphi_{Z_n}(t)
=
\left[
\varphi\left(
\frac{t}{\sqrt n}
\right)
\right]^n.
$$

Hence

$$
\varphi_{Z_n}(t)
\to
e^{-t^2/2}.
$$

Lévy's continuity theorem then converts the transform limit into

$$
Z_n\xrightarrow{d}N(0,1).
$$


In [ ]:
clt_n = widgets.IntSlider(value=20, min=1, max=500, description="n")
clt_t = widgets.FloatSlider(value=1.0, min=-4, max=4, step=0.1, description="t")
clt_output = widgets.Output()


def update_rademacher_cf_clt(*_):
    with clt_output:
        clear_output(wait=True)

        n = clt_n.value
        t = clt_t.value

        # Standardized sum of iid Rademacher variables:
        # phi_n(t) = cos(t/sqrt(n))^n.
        phi_n = math.cos(t/math.sqrt(n))**n
        phi_normal = math.exp(-t*t/2)

        display(Math(r"\varphi_{Z_n}(t)=" + f"{phi_n:.10f}"))
        display(Math(r"e^{-t^2/2}=" + f"{phi_normal:.10f}"))
        display(Math(r"\text{absolute difference}=" + f"{abs(phi_n-phi_normal):.10f}"))


for control in (clt_n,clt_t):
    control.observe(update_rademacher_cf_clt, names="value")

display(widgets.VBox([
    widgets.HBox([clt_n,clt_t]),
    clt_output,
]))
update_rademacher_cf_clt()


### Why the normal distribution appears

After centering and variance normalization, only the quadratic term

$$
-\frac{u^2}{2}
$$

survives in the characteristic-function limit.

This produces

$$
e^{-t^2/2},
$$

the characteristic function of the standard normal law.

The detailed shape of the parent distribution disappears.


### Finite variance matters

The classical CLT in this chapter requires a finite, non-zero variance.

Heavy-tailed variables can have very different fluctuation behavior.

The standard Cauchy law is the main warning example.


## 18. Studentized Central Limit Theorem

Under the assumptions of the classical CLT, let

$$
\widehat\sigma_n^2
=
\frac1{n-1}
\sum_{k=1}^n
(X_k-\overline X_n)^2.
$$

Then

$$
\boxed{
\frac{
\sqrt n(\overline X_n-\mu)
}{
\widehat\sigma_n
}
\xrightarrow{d}
N(0,1).
}
$$

The result follows by combining:

1. the CLT;
2. strong consistency $\widehat\sigma_n\to\sigma$;
3. Slutsky's theorem.


In [ ]:
stud_n = widgets.IntSlider(value=30, min=5, max=300, description="n")
stud_R = widgets.IntSlider(value=20000, min=1000, max=100000, step=1000, description="reps")
stud_output = widgets.Output()


def update_studentized(*_):
    with stud_output:
        clear_output(wait=True)

        n = stud_n.value
        R = stud_R.value

        # Parent law: exponential(1), mean=1, variance=1.
        rng = np.random.default_rng(2026)
        x = rng.exponential(scale=1,size=(R,n))

        means = x.mean(axis=1)
        s = x.std(axis=1,ddof=1)
        z = math.sqrt(n)*(means-1)/s

        empirical = np.mean(np.abs(z)<=1.96)

        display(Math(
            r"\widehat P(|T_n|\le1.96)=" + f"{empirical:.6f}"
        ))
        display(Markdown("Standard-normal benchmark: approximately **0.95**"))


for control in (stud_n,stud_R):
    control.observe(update_studentized, names="value")

display(widgets.VBox([
    widgets.HBox([stud_n,stud_R]),
    stud_output,
]))
update_studentized()


## 19. Quantitative refinement: Berry--Esseen

The classical CLT is qualitative. By itself it does not say how large $n$ must be.

If, in addition,

$$
\rho_3
=
\mathbb E|X_1-\mu|^3
<
\infty,
$$

then Berry--Esseen gives a universal constant $C<\infty$ such that

$$
\boxed{
\sup_x
\left|
P\left(
\frac{\sqrt n(\overline X_n-\mu)}{\sigma}
\le x
\right)
-
\Phi(x)
\right|
\le
\frac{
C\rho_3
}{
\sigma^3\sqrt n
}.
}
$$

The theorem is stated in the chapter as a quantitative refinement and is not proved there.


The conceptual difference is:

$$
\boxed{
\text{CLT}
=
\text{convergence}
}
$$

while

$$
\boxed{
\text{Berry--Esseen}
=
\text{an explicit }O(n^{-1/2})\text{ rate under an extra third-moment assumption}.
}
$$


## 20. The binomial case revisited

If

$$
B_n\sim\operatorname{Bin}(n,p),
\qquad
0<p<1,
$$

then

$$
\boxed{
\frac{
B_n-np
}{
\sqrt{np(1-p)}
}
\xrightarrow{d}
N(0,1).
}
$$

This follows by writing $B_n$ as a sum of independent Bernoulli variables.


### Continuity correction

For large $n$,

$$
P(a\le B_n\le b)
$$

is approximated by

$$
\boxed{
\Phi\left(
\frac{b+1/2-np}{\sqrt{np(1-p)}}
\right)
-
\Phi\left(
\frac{a-1/2-np}{\sqrt{np(1-p)}}
\right).
}
$$

This remains an approximation, not an identity.


In [ ]:
bin_n = widgets.IntSlider(value=100, min=10, max=1000, step=10, description="n")
bin_p = widgets.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, description="p")
bin_width = widgets.FloatSlider(value=2, min=0.5, max=3, step=0.1, description="sd width")
bin_output = widgets.Output()


def update_binomial_clt(*_):
    with bin_output:
        clear_output(wait=True)

        n = bin_n.value
        p = bin_p.value
        width = bin_width.value

        mu = n*p
        sigma = math.sqrt(n*p*(1-p))

        a = math.ceil(mu-width*sigma)
        b = math.floor(mu+width*sigma)

        exact = binomial_interval_probability(a,b,n,p)

        approx = (
            normal_cdf((b+0.5-mu)/sigma)
            - normal_cdf((a-0.5-mu)/sigma)
        )

        display(Markdown(f"Integer interval: **[{a},{b}]**"))
        display(Math(r"\text{exact binomial probability}=" + f"{exact:.8f}"))
        display(Math(r"\text{normal approximation}=" + f"{approx:.8f}"))
        display(Math(r"\text{absolute error}=" + f"{abs(exact-approx):.8f}"))


for control in (bin_n,bin_p,bin_width):
    control.observe(update_binomial_clt, names="value")

display(widgets.VBox([
    widgets.HBox([bin_n,bin_p,bin_width]),
    bin_output,
]))
update_binomial_clt()


## 21. Python laboratory: seeing the laws of large numbers and the CLT

The chapter separates three numerical pictures:

1. **Strong law:** one running sample path stabilizes;
2. **Weak law:** a fixed deviation event becomes less probable as $n$ grows;
3. **CLT:** repeated standardized sums become approximately standard normal.

The next experiments keep these ideas separate.


### Weak-law experiment for Bernoulli$(0.3)$

We estimate

$$
P\left(
\left|
\frac{N_n}{n}
-
0.3
\right|
>
0.05
\right)
$$

for several sample sizes and compare the result with the Chebyshev bound.


In [ ]:
weak_ns = [50,200,1000,5000]
p = 0.3
eps = 0.05
R = 30000
rng = np.random.default_rng(2026)

rows = []

for n in weak_ns:
    counts = rng.binomial(n,p,size=R)
    freq = counts/n
    empirical = np.mean(np.abs(freq-p)>eps)
    bound = min(1,p*(1-p)/(n*eps*eps))

    rows.append(
        f"| {n} | {empirical:.6f} | {bound:.6f} |"
    )

display(Markdown(
    "| n | empirical deviation probability | Chebyshev bound |\n"
    "|---:|---:|---:|\n"
    + "\n".join(rows)
))


### CLT from a non-normal parent: exponential variables

Let

$$
X_i\sim\operatorname{Exp}(1),
$$

so

$$
\mu=1,
\qquad
\sigma=1.
$$

The standardized sum is

$$
Z_n
=
\frac{
X_1+\cdots+X_n-n
}{
\sqrt n
}.
$$


In [ ]:
clt_sim_n = widgets.IntSlider(value=20, min=1, max=200, description="n")
clt_sim_R = widgets.IntSlider(value=30000, min=1000, max=100000, step=1000, description="reps")
clt_sim_output = widgets.Output()


def update_clt_hist(*_):
    with clt_sim_output:
        clear_output(wait=True)

        n = clt_sim_n.value
        R = clt_sim_R.value

        rng = np.random.default_rng(2026)
        x = rng.exponential(scale=1,size=(R,n))
        z = standardized_sum(x,1,1)

        grid = np.linspace(-4,4,700)

        fig, ax = plt.subplots(figsize=(8,3.6))
        ax.hist(z,bins=60,density=True,alpha=0.35)
        ax.plot(grid,normal_pdf(grid))
        ax.set_xlabel("standardized sum")
        ax.set_ylabel("density")
        ax.set_title("CLT from an exponential parent distribution")
        plt.show()

        empirical = np.mean(np.abs(z)<=1.96)
        display(Math(r"\widehat P(|Z_n|\le1.96)=" + f"{empirical:.6f}"))


for control in (clt_sim_n,clt_sim_R):
    control.observe(update_clt_hist, names="value")

display(widgets.VBox([
    widgets.HBox([clt_sim_n,clt_sim_R]),
    clt_sim_output,
]))
update_clt_hist()


### Empirical cdf at $-1,0,1$

For standardized sums, convergence in distribution predicts that

$$
P(Z_n\le x)
$$

should approach $\Phi(x)$ at every real $x$.


In [ ]:
cdf_ns = [1,5,20,100]
R = 50000
rng = np.random.default_rng(2026)

rows = []
for n in cdf_ns:
    x = rng.uniform(0,1,size=(R,n))
    mu = 0.5
    sigma = math.sqrt(1/12)
    z = standardized_sum(x,mu,sigma)

    vals = [empirical_cdf_at(z,a) for a in [-1,0,1]]

    rows.append(
        f"| {n} | {vals[0]:.4f} | {vals[1]:.4f} | {vals[2]:.4f} |"
    )

display(Markdown(
    "| n | empirical F(-1) | empirical F(0) | empirical F(1) |\n"
    "|---:|---:|---:|---:|\n"
    + "\n".join(rows)
))

display(Markdown(
    f"Standard normal values: "
    f"$\\Phi(-1)={normal_cdf(-1):.4f}$, "
    f"$\\Phi(0)=0.5000$, "
    f"$\\Phi(1)={normal_cdf(1):.4f}$."
))


## 22. Cauchy contrast: averages do not stabilize

Let $X_1,\ldots,X_n$ be independent standard Cauchy variables.

Chapter 13 established

$$
\varphi_X(t)=e^{-|t|}.
$$

Therefore

$$
\varphi_{\overline X_n}(t)
=
\left[
\varphi_X\left(
\frac tn
\right)
\right]^n
=
\left[
e^{-|t|/n}
\right]^n
=
e^{-|t|}.
$$

By uniqueness,

$$
\boxed{
\overline X_n
\stackrel d=
X_1
\quad\text{for every }n.
}
$$

Averaging does not concentrate the distribution.


This does not contradict the LLN because the standard Cauchy distribution has no finite mean.

It also lies outside the classical finite-variance CLT.


In [ ]:
cauchy_ns = [1,10,100,1000]
R = 20000
rng = np.random.default_rng(2026)

rows = []

for n in cauchy_ns:
    x = rng.standard_cauchy(size=(R,n))
    means = x.mean(axis=1)
    q = np.quantile(means,[0.25,0.5,0.75])

    rows.append(
        f"| {n} | {q[0]:.4f} | {q[1]:.4f} | {q[2]:.4f} |"
    )

display(Markdown(
    "| n | q25 of sample mean | median | q75 |\n"
    "|---:|---:|---:|---:|\n"
    + "\n".join(rows)
))


## 23. Exact normality versus asymptotic normality

If

$$
X_i\stackrel{\mathrm{i.i.d.}}{\sim}N(\mu,\sigma^2),
$$

then for every $n$,

$$
S_n
\sim
N(n\mu,n\sigma^2).
$$

Therefore

$$
\boxed{
\frac{S_n-n\mu}{\sigma\sqrt n}
\sim
N(0,1)
}
$$

**exactly for every $n$**.

The CLT conclusion is asymptotic in general, but exact in the Gaussian parent case.


## 24. Solved-style computations


### Chebyshev bound for a sample mean

Suppose $X_1,\ldots,X_{100}$ are independent with common mean $5$ and variance $9$.

Then

$$
\operatorname{Var}(\overline X_{100})
=
\frac9{100}.
$$

Therefore

$$
\boxed{
P(
|\overline X_{100}-5|
\ge1
)
\le
0.09.
}
$$


In [ ]:
display(Math(
    r"P(|\overline X_{100}-5|\ge1)\le"
    + f"{chebyshev_mean_bound(9,100,1):.2f}"
))


### Poisson sum approximation

Let $X_1,\ldots,X_{100}$ be i.i.d. Poisson$(4)$ and

$$
S_{100}
=
X_1+\cdots+X_{100}.
$$

Then

$$
\mathbb E[S_{100}]=400,
\qquad
\operatorname{SD}(S_{100})=20.
$$

With continuity correction,

$$
P(S_{100}\le430)
\approx
\Phi\left(
\frac{430.5-400}{20}
\right).
$$


In [ ]:
z = (430.5-400)/20
display(Math(r"z=" + f"{z:.4f}"))
display(Math(r"\Phi(z)=" + f"{normal_cdf(z):.8f}"))


### Sum of $64$ uniform variables

If

$$
X_i\sim U(0,1),
$$

then

$$
\mu=\frac12,
\qquad
\sigma^2=\frac1{12}.
$$

For

$$
S_{64},
$$

$$
\mathbb E[S_{64}]=32,
$$

and

$$
\operatorname{SD}(S_{64})
=
\sqrt{\frac{64}{12}}
=
\frac4{\sqrt3}.
$$

The CLT approximates

$$
P(30\le S_{64}\le34)
$$

by the corresponding normal interval.


In [ ]:
sd = math.sqrt(64/12)
approx = normal_cdf((34-32)/sd)-normal_cdf((30-32)/sd)

display(Math(r"\operatorname{SD}(S_{64})=" + f"{sd:.6f}"))
display(Math(r"\text{CLT approximation}=" + f"{approx:.8f}"))


## 25. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random","random"),
        ("Modes of convergence","modes"),
        ("Chebyshev","cheb"),
        ("Weak law","wlln"),
        ("Bernoulli frequency","bern"),
        ("Borel-Cantelli","bc"),
        ("Strong law","slln"),
        ("Slutsky","slutsky"),
        ("CLT","clt"),
        ("Cauchy contrast","cauchy"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}


def make_exercise(_=None):
    kind = exercise_kind.value

    if kind == "random":
        kind = exercise_rng.choice([
            "modes","cheb","wlln","bern","bc","slln",
            "slutsky","clt","cauchy"
        ])

    if kind == "modes":
        target = "yes"
        prompt = "Does convergence in probability always imply convergence in distribution? yes/no"
        hint = "This is one of the basic one-way arrows."
        solution = r"\text{Yes.}"

    elif kind == "cheb":
        target = "0.09"
        prompt = "Var(X_i)=9 and n=100. Use Chebyshev to bound P(|Xbar-mean|>=1)."
        hint = "Var(Xbar)=9/100."
        solution = r"P(|\overline X-\mu|\ge1)\le0.09."

    elif kind == "wlln":
        target = "0"
        prompt = "Under the finite-variance WLLN, what is the probability limit of Xbar_n-mu?"
        hint = "The sample mean converges to mu in probability."
        solution = r"\overline X_n-\mu\xrightarrow{P}0."

    elif kind == "bern":
        target = "2000"
        prompt = "Using the universal Bernoulli-Chebyshev bound, find a sufficient n for epsilon=0.05 and eta=0.05."
        hint = "Use 1/(4 epsilon^2 eta)."
        solution = r"n\ge2000."

    elif kind == "bc":
        target = "no"
        prompt = "Does the first Borel-Cantelli lemma require independence? yes/no"
        hint = "Summability alone is sufficient."
        solution = r"\text{No.}"

    elif kind == "slln":
        target = "yes"
        prompt = "Can the i.i.d. strong law hold when variance is infinite but E|X| is finite? yes/no"
        hint = "The chapter's SLLN assumes only a finite first absolute moment."
        solution = r"\text{Yes.}"

    elif kind == "slutsky":
        target = "yes"
        prompt = "If X_n=>X and Y_n->2 in probability, does X_n/Y_n=>X/2? yes/no"
        hint = "Use the ratio part of Slutsky with nonzero constant limit."
        solution = r"\text{Yes.}"

    elif kind == "clt":
        target = "sqrt(n)"
        prompt = "What is the natural fluctuation scale in the classical CLT? Enter sqrt(n)."
        hint = "Var(S_n-nmu)=n sigma^2."
        solution = r"\sqrt n."

    else:
        target = "yes"
        prompt = "For iid standard Cauchy variables, does Xbar_n have the same distribution as X_1 for every n? yes/no"
        hint = "Use the characteristic function e^{-|t|}."
        solution = r"\text{Yes.}"

    state.clear()
    state.update(target=target,hint=hint,solution=solution)
    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))

    with feedback_output:
        clear_output(wait=True)


def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + state["hint"]))


def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))


def check(_):
    with feedback_output:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ","")
        target = state["target"].strip().lower().replace(" ","")

        correct = guess == target

        if not correct:
            try:
                correct = abs(float(guess)-float(target)) < 5e-4
            except Exception:
                pass

        display(Markdown(
            "**Correct.**"
            if correct
            else "**Not yet. Check the convergence mode and theorem assumptions first.**"
        ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind,new_button]),
    prompt_output,
    widgets.HBox([answer_box,check_button]),
    widgets.HBox([hint_button,reveal_button]),
    feedback_output,
]))

make_exercise()


## 26. AI Audit: limit-theorem claims

Use this checklist on any AI-generated solution.

1. Is the mode of convergence stated explicitly?
2. Is convergence in distribution incorrectly treated as pathwise convergence?
3. Is a converse implication being used without justification?
4. If the limit is constant, is the special distribution-to-probability converse recognized?
5. Is Chebyshev used only when the relevant variance is finite?
6. Is independence used correctly when computing the variance of a sum?
7. Is the weak law confused with eventual exact equality of the sample mean and population mean?
8. Is Bernoulli's theorem interpreted as frequency stabilization rather than finite-time equality?
9. Is a Chebyshev sample-size bound presented as sufficient rather than minimal?
10. Is the first Borel--Cantelli lemma incorrectly said to require independence?
11. Is the second Borel--Cantelli lemma used without independence?
12. Is convergence in probability incorrectly upgraded to almost-sure convergence?
13. Does a Kolmogorov maximal-inequality argument use independent mean-zero variables with finite variances?
14. Is the Kolmogorov convergence criterion applied only when the variance series is summable?
15. Is the strong law correctly stated under $\mathbb E|X_1|<\infty$?
16. Is infinite variance incorrectly claimed to destroy every law of large numbers?
17. Is the finite-variance classical CLT applied to an infinite-variance parent law?
18. Is sample-variance consistency derived using the strong law for both $X_i$ and $X_i^2$?
19. Is Lévy's continuity theorem used after establishing pointwise convergence to a genuine characteristic function?
20. Is Slutsky's theorem used with one factor converging in probability to a constant?
21. Is the CLT applied to the **centered and $\sqrt n$-scaled** sum?
22. Is the unscaled sample mean incorrectly said to converge to a non-degenerate normal law?
23. Is the local characteristic-function expansion used with a finite second moment?
24. Is the role of independence in the CF product explicitly identified?
25. Is the standard normal characteristic function $e^{-t^2/2}$ recognized as the transform limit?
26. Is studentization justified by sample-variance consistency and Slutsky?
27. Is Berry--Esseen distinguished from the classical CLT?
28. Is the continuity correction treated as an approximation device, not an exact identity?
29. Is the Gaussian-parent case recognized as exactly normal for every $n$?
30. Is the Cauchy average recognized as having the same law as one summand?
31. Is the absence of finite Cauchy mean/variance used to explain why the chapter's LLN/CLT hypotheses fail?
32. Is convergence in distribution being incorrectly used to infer convergence of expectations without extra assumptions?
33. Is simulation clearly separated from proof?

### Three chapter AI-audit claims

1. “If a distribution has infinite variance, the law of large numbers fails.”
2. “If $X_n\xrightarrow{d}X$, then $\mathbb E[X_n]\to\mathbb E[X]$ whenever all displayed expectations are finite.”
3. “The central limit theorem says that $\overline X_n$ converges in distribution to a normal random variable.”

All three statements are false as written.


### Correcting the three claims

**Claim 1:** false. The strong law needs only

$$
\mathbb E|X_1|<\infty.
$$

The Pareto example in this chapter has finite mean and infinite variance, yet

$$
\overline X_n\to3
$$

almost surely.

**Claim 2:** false. Convergence in distribution alone does not control expectations. For example, let

$$
X_n=
\begin{cases}
n,&\text{with probability }1/n,\\
0,&\text{otherwise}.
\end{cases}
$$

Then

$$
X_n\xrightarrow{d}0,
$$

but

$$
\mathbb E[X_n]=1
$$

for every $n$.

**Claim 3:** false. The LLN gives

$$
\overline X_n\to\mu,
$$

while the CLT gives a normal limit only after centering and multiplying by $\sqrt n$:

$$
\frac{\sqrt n(\overline X_n-\mu)}{\sigma}
\xrightarrow{d}
N(0,1).
$$


### Suggested AI-guided activities

- “Act as a Socratic tutor on the law of large numbers. Make me derive the mean and variance of the sample mean before using Chebyshev.”
- “Give me examples of all four convergence modes and make me identify which implication arrows are valid.”
- “Make me construct a sequence that converges in probability but not almost surely using Borel--Cantelli.”
- “Guide me through the strong-law proof architecture: truncation, Borel--Cantelli, Kolmogorov, Kronecker and Cesàro.”
- “Give me a finite-mean infinite-variance Pareto example and make me classify which LLN and CLT results apply.”
- “Guide me through the characteristic-function proof of the CLT one step at a time.”
- “Give me a studentized statistic and make me identify exactly where sample-variance consistency and Slutsky are used.”
- “Compare a qualitative CLT statement with a Berry--Esseen rate statement.”
- “Simulate standard Cauchy averages and require me to explain the result using the characteristic function, not by visual impression alone.”


## 27. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. Almost-sure convergence always implies convergence in probability:",
        ["Choose...","true","false"],
        "true",
        r"X_n\xrightarrow{\mathrm{a.s.}}X\Longrightarrow X_n\xrightarrow{P}X.",
    ),
    (
        "2. Convergence in distribution always implies convergence in probability:",
        ["Choose...","true","false"],
        "false",
        r"\text{The converse fails unless, for example, the limit is constant.}",
    ),
    (
        "3. L^p convergence implies convergence in probability:",
        ["Choose...","true","false"],
        "true",
        r"\text{Use Markov on }|X_n-X|^p.",
    ),
    (
        "4. Chebyshev requires finite variance:",
        ["Choose...","true","false"],
        "true",
        r"P(|X-\mu|\ge\varepsilon)\le\sigma^2/\varepsilon^2.",
    ),
    (
        "5. The weak law says the sample mean becomes exactly equal to mu after finite time:",
        ["Choose...","true","false"],
        "false",
        r"\text{It states convergence in probability.}",
    ),
    (
        "6. The first Borel-Cantelli lemma requires independence:",
        ["Choose...","true","false"],
        "false",
        r"\sum P(A_n)<\infty\Longrightarrow P(A_n\text{ i.o.})=0.",
    ),
    (
        "7. The second Borel-Cantelli lemma uses independence:",
        ["Choose...","true","false"],
        "true",
        r"\text{The divergent-sum converse requires independence in this theorem.}",
    ),
    (
        "8. The strong law in this chapter requires finite variance:",
        ["Choose...","true","false"],
        "false",
        r"\mathbb E|X_1|<\infty\text{ is sufficient for the iid SLLN.}",
    ),
    (
        "9. The classical CLT uses a sqrt(n) fluctuation scale:",
        ["Choose...","true","false"],
        "true",
        r"\frac{S_n-n\mu}{\sigma\sqrt n}\Rightarrow N(0,1).",
    ),
    (
        "10. Slutsky allows a consistent random scale to replace a constant asymptotically:",
        ["Choose...","true","false"],
        "true",
        r"\text{This is the basis of studentization.}",
    ),
    (
        "11. Berry-Esseen gives an explicit convergence-rate bound under an extra third-moment assumption:",
        ["Choose...","true","false"],
        "true",
        r"\text{Its error bound is of order }n^{-1/2}.",
    ),
    (
        "12. Standard Cauchy sample means have the same distribution as one observation:",
        ["Choose...","true","false"],
        "true",
        r"\varphi_{\overline X_n}(t)=e^{-|t|}.",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="500px"),
    )
    quiz_widgets.append(dropdown)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:700px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_,_,correct,_) in zip(quiz_widgets,quiz_data)
        )

        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))

        for i,(widget,(_,_,correct,explanation)) in enumerate(
            zip(quiz_widgets,quiz_data),1
        ):
            mark = "✓" if widget.value == correct else "✗"
            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))
            display(Math(explanation))


grade_button.on_click(grade_quiz)

display(widgets.VBox(
    quiz_rows+[grade_button,quiz_output]
))


## 28. Automatic mathematical verification

The final code cell checks representative formulas from the chapter.


In [ ]:
# Chebyshev sample mean example.
assert abs(chebyshev_mean_bound(9,100,1)-0.09) < 1e-15

# Bernoulli universal sample size.
assert bernoulli_universal_sample_size(0.05,0.05) == 2000

# WLLN Poisson bound.
assert abs(chebyshev_mean_bound(3,100,0.5)-0.12) < 1e-15

# Bernoulli specific bound is bounded by universal bound.
p = 0.3
n = 1000
eps = 0.05

specific = p*(1-p)/(n*eps*eps)
universal = 1/(4*n*eps*eps)

assert specific <= universal+1e-15

# Cauchy sample-mean characteristic function identity.
for n in [1,2,10,100]:
    for t in [-3,-1,0,0.4,2]:
        lhs = standard_cauchy_cf(t/n)**n
        rhs = standard_cauchy_cf(t)
        assert abs(lhs-rhs) < 1e-12

# Rademacher CLT CF convergence for increasing n at a fixed t.
t = 1.0
errors = []
for n in [10,50,200,1000]:
    phi_n = math.cos(t/math.sqrt(n))**n
    errors.append(abs(phi_n-math.exp(-t*t/2)))

assert errors[-1] < errors[0]

# Exact normal interval values.
assert abs(normal_cdf(0)-0.5) < 1e-15
assert abs(normal_cdf(1)-(1-normal_cdf(-1))) < 1e-15

# Exact normal parent: standardized sum remains N(0,1) by parameter algebra.
mu = 3.0
sigma = 2.0
for n in [1,2,10,100]:
    sum_mean = n*mu
    sum_var = n*sigma*sigma

    standardized_mean = (sum_mean-n*mu)/(sigma*math.sqrt(n))
    standardized_var = sum_var/(sigma*sigma*n)

    assert abs(standardized_mean) < 1e-15
    assert abs(standardized_var-1) < 1e-15

# Pareto shape 3/2 has mean 3 and infinite second moment.
alpha = 1.5
pareto_mean = alpha/(alpha-1)
assert abs(pareto_mean-3) < 1e-15
assert alpha < 2

# Binomial CLT continuity correction produces valid probability.
n,p = 100,0.5
a,b = 40,60
exact = binomial_interval_probability(a,b,n,p)
mu = n*p
sigma = math.sqrt(n*p*(1-p))
approx = normal_cdf((b+0.5-mu)/sigma)-normal_cdf((a-0.5-mu)/sigma)

assert 0 <= exact <= 1
assert 0 <= approx <= 1
assert abs(exact-approx) < 0.01

# Studentized variance formula positive in representative sample.
rng = np.random.default_rng(123)
sample = rng.exponential(scale=1,size=1000)
assert sample_variance_unbiased(sample) > 0

# Poisson CF convergence with lambda_n -> lambda.
t = 0.8
target = np.exp(2*(np.exp(1j*t)-1))
seq = [
    np.exp((2+1/n)*(np.exp(1j*t)-1))
    for n in [1,10,100,1000]
]
assert abs(seq[-1]-target) < abs(seq[0]-target)

show_result(
    "All Chapter 14 automatic checks passed",
    r"X_n\xrightarrow{\mathrm{a.s.}}X\Longrightarrow X_n\xrightarrow{P}X\Longrightarrow X_n\xrightarrow{d}X",
    r"P(|\overline X_n-\mu|>\varepsilon)\le\frac{\sigma^2}{n\varepsilon^2}",
    r"\overline X_n\xrightarrow{\mathrm{a.s.}}\mu\quad\text{when }\mathbb E|X_1|<\infty",
    r"\frac{S_n-n\mu}{\sigma\sqrt n}\xrightarrow{d}N(0,1)",
    r"\varphi_{\overline C_n}(t)=e^{-|t|}",
    note=(
        "Chebyshev, Bernoulli sample-size, Cauchy stability, CLT transform, "
        "binomial approximation and Poisson-CF checks all passed."
    ),
)


## 29. Chapter map

| Chapter concept | Computational representation |
|---|---|
| LLN versus CLT | shrinking mean SD versus $\sqrt n$ fluctuation scale |
| convergence in probability | fixed-error probability |
| convergence in distribution | cdf convergence |
| almost sure convergence | pathwise viewpoint |
| $L^p$ convergence | mean error size |
| implication diagram | one-way arrows |
| a.s. not $L^1$ | moving-spike example |
| distribution not probability | alternating-sign example |
| Chebyshev | universal versus exact normal tail |
| finite-variance WLLN | Poisson sample averages |
| variance criterion | non-i.i.d. independent averages |
| Bernoulli theorem | relative-frequency concentration |
| historical frequency problem | sufficient sample-size calculator |
| first Borel--Cantelli | summable bad-event probabilities |
| second Borel--Cantelli | independent divergent-sum recurrence |
| probability not almost sure | indicator counterexample |
| Kolmogorov maximal inequality | random-walk path maximum |
| Kolmogorov convergence criterion | random harmonic series |
| Cesàro/Kronecker | deterministic averaging tools |
| strong law | Bernoulli running sample path |
| finite mean, infinite variance | Pareto example |
| sample variance consistency | Bernoulli variance path |
| Scheffé | nearby normal densities |
| Lévy continuity theorem | Poisson parameters converging |
| Slutsky | consistent replacement of constants |
| local CF expansion | Rademacher Taylor check |
| complex exponentiation limit | final analytic CLT step |
| classical CLT | CF convergence to $e^{-t^2/2}$ |
| studentized CLT | empirical exponential experiment |
| Berry--Esseen | rate versus qualitative convergence |
| binomial normal limit | continuity-corrected approximation |
| Python laboratory | WLLN, SLLN and CLT separated |
| Cauchy contrast | invariant distribution of sample means |
| exact Gaussian case | standard normal for every $n$ |
| AI Audit | assumptions and convergence-mode checks |

The main hierarchy is:

$$
\boxed{
\text{stabilization of averages}
\neq
\text{shape of scaled fluctuations}.
}
$$

The law of large numbers answers the first question.

The central limit theorem answers the second.
